# 根据已完成方法的源表更新比较图

只重绘当前完整分数表中已有的方法，使用同一份缓存分布和 UMAP 坐标。删去方法不再进入图例或面板；其余方法的源数据与五项分数不变。新增方法必须完成真实评分并保存绘图源表后才能进入比较。

In [ ]:
from pathlib import Path
import os,sys
import numpy as np
import pandas as pd
project=Path.cwd()
if not (project/'data/paths.py').exists():project=project.parent
sys.path.insert(0,str(project))
from data.paths import OUTPUTS,check_output
from evaluate.plotting import plt,plot_name,plot_color,save_figure
from evaluate.A1.plotting import draw_marginal,marginal_legend
from evaluate.A2.plotting import plot_duplicate_pearson
from evaluate.A3.plotting import plot_joint_umap

def redraw_comparison(output_root):
    output_root=check_output(output_root)
    source=output_root/'internal/evaluation/liu2023/full'
    scores=pd.read_csv(output_root/'tables/liu2023全数据集.csv')
    methods=scores['方法'].tolist()
    regions=pd.read_csv(source/'A1_plot_regions.csv')
    pooled=pd.read_csv(source/'A1_plot_histograms.csv')
    typed=pd.read_csv(source/'A1_type_histograms.csv')
    edges=np.linspace(0,1,31)
    def histogram(table,region,method,level=None):
        rows=table.loc[table['原region'].eq(region)&table['方法'].eq(method)]
        if level is not None:rows=rows.loc[rows['细胞类型'].eq(level)]
        rows=rows.sort_values('分箱左界')
        assert len(rows)==30,(region,method,level)
        return rows['概率'].to_numpy(),float(rows['越界比例'].iloc[0]),int(rows['有限值数'].iloc[0])
    figure,axes=plt.subplots(len(regions),len(methods),figsize=(3.15*len(methods),2*len(regions)+.7),sharex=True,sharey='row',squeeze=False)
    figure.subplots_adjust(left=.075,right=.995,bottom=.85/figure.get_figheight(),top=1-.95/figure.get_figheight(),wspace=.1,hspace=.34)
    for i,region in regions.iterrows():
        hist={name:histogram(pooled,region['原region'],name) for name in ['真实数据',*methods]}
        upper=max(.1,np.ceil(max(value[0].max() for value in hist.values())*12)/10)
        for j,method in enumerate(methods):
            axis=axes[i,j]
            draw_marginal(axis,hist['真实数据'],hist[method],method,edges)
            axis.text(.02,.95,region['原region'],transform=axis.transAxes,fontsize=6,va='top',color='#555555')
            axis.set_ylim(0,upper)
            if i==0:axis.set_title(plot_name(method),color=plot_color(method),fontsize=10,weight='bold',pad=7)
            if i==len(regions)-1:axis.set_xlabel('Methylation level',fontsize=8)
            if j==0:axis.set_ylabel(f"{region['展示名称']}\nProbability per bin",fontsize=8)
    marginal_legend(figure,methods,.98)
    figure.suptitle('Literature-guided Tle4 locus windows (mm10)',fontsize=11,weight='bold',y=1.01)
    figure.text(.075,.015,'All cell types pooled. Shared 30 bins; finite observations only.',fontsize=7,color='#555555')
    save_figure(figure,output_root,'A1_marginal_regions')
    detail=regions.loc[regions['单区域展示'].eq(1)].iloc[0]
    levels=detail['展示细胞类型'].split('|')
    hist={(name,level):histogram(typed,detail['原region'],name,level) for name in ['真实数据',*methods] for level in levels}
    upper=max(.1,np.ceil(max(value[0].max() for value in hist.values())*12)/10)
    figure,axes=plt.subplots(len(methods),len(levels),figsize=(3*len(levels),1.9*len(methods)+1),sharex=True,sharey=True,squeeze=False)
    figure.subplots_adjust(left=.09,right=.995,bottom=.85/figure.get_figheight(),top=1-1.35/figure.get_figheight(),wspace=.12,hspace=.24)
    for i,method in enumerate(methods):
        for j,level in enumerate(levels):
            axis=axes[i,j]
            real,synthetic=hist['真实数据',level],hist[method,level]
            draw_marginal(axis,real,synthetic,method,edges)
            axis.text(.02,.95,f'n: R={real[2]}, S={synthetic[2]}',transform=axis.transAxes,fontsize=6.5,va='top',color='#555555')
            axis.set_ylim(0,upper)
            if i==0:axis.set_title(level.replace(' CTX Glut','\nCTX Glut'),fontsize=9,pad=8)
            if j==0:axis.set_ylabel(f'{plot_name(method)}\nProbability per bin',color=plot_color(method),fontsize=8)
            if i==len(methods)-1:axis.set_xlabel('Methylation level',fontsize=8)
    marginal_legend(figure,methods,1-.48/figure.get_figheight())
    figure.suptitle(f"{detail['展示名称']} | {detail['原region']}\nSingle-region distributions by Subclass",fontsize=11,weight='bold',y=1.01)
    figure.text(.09,.015,'Rows: methods. Columns: Subclass. R = real; S = synthetic; n counts finite observations. Shared 30 bins and y-axis.',fontsize=7,color='#555555')
    save_figure(figure,output_root,'A1_region_by_type')
    correlations=pd.read_csv(source/'A2_duplicate_correlations.csv')
    correlations=correlations.loc[correlations['方法'].isin(['真实数据',*methods])]
    summary=correlations.groupby('方法',sort=False).agg(配对数=('Pearson','size'),有效配对数=('Pearson','count'),Pearson均值=('Pearson','mean')).reset_index()
    summary=summary.set_index('方法').loc[['真实数据',*methods]].reset_index()
    for method in methods:summary.loc[summary['方法'].eq(method),'Pearson均值']=float(scores.loc[scores['方法'].eq(method),'A2 Pearson↑'].iloc[0])
    plot_duplicate_pearson(correlations,summary,output_root)
    coordinates=pd.read_csv(source/'A3_joint_coordinates.csv')
    coordinates=coordinates.loc[coordinates['方法'].isin(methods)]
    assert set(coordinates['方法'])==set(methods)
    plot_joint_umap(coordinates,scores.rename(columns={'A3 miLISI↑':'miLISI'}).assign(近邻数=30),methods,output_root,2022)
    print('比较图更新完成',methods)


In [ ]:
redraw_comparison(Path(os.environ.get('COMPARISON_OUTPUT_ROOT',str(OUTPUTS))))